# Melodiva AI — ACE-Step GPU lab
Select a GPU runtime. This notebook runs interactive inference, not a public API tunnel. For persistent remote Melodiva integration use a dedicated authenticated HTTPS GPU host.


In [ ]:
import subprocess, pathlib, os, time, json, urllib.request
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv"], check=True)
root = pathlib.Path("/content/ACE-Step-1.5")
if not root.exists():
    subprocess.run(["git", "clone", "https://github.com/ace-step/ACE-Step-1.5.git", str(root)], check=True)
subprocess.run(["uv", "sync", "--python", "3.11"], cwd=root, check=True)


In [ ]:
log = open("/content/acestep.log", "w")
server = subprocess.Popen(["uv", "run", "acestep-api", "--host", "127.0.0.1", "--port", "8001"], cwd=root, stdout=log, stderr=subprocess.STDOUT)
base = "http://127.0.0.1:8001"
def api(path, payload=None):
    req = urllib.request.Request(base+path, data=None if payload is None else json.dumps(payload).encode(), headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        result = json.load(r)
    if result.get("code",200)!=200 or result.get("error"):
        raise RuntimeError(result)
    return result.get("data")
for _ in range(360):
    if server.poll() is not None:
        raise RuntimeError(pathlib.Path("/content/acestep.log").read_text()[-5000:])
    try:
        api("/health")
        break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError("Check /content/acestep.log")
print("Engine reachable")


In [ ]:
prompt = "Bongo Flava, warm guitar, melodic bass, energetic percussion" # @param {type:"string"}
lyrics = "[inst]" # @param {type:"string"}
duration = 30 # @param {type:"integer"}
task = api("/release_task", {"prompt":prompt,"lyrics":lyrics,"audio_duration":duration,"audio_format":"wav","bpm":110,"thinking":False})
for _ in range(450):
    item = api("/query_result", {"task_id_list":[task["task_id"]]})[0]
    if item["status"] == 2:
        raise RuntimeError(item)
    if item["status"] == 1:
        result = item["result"]
        result = json.loads(result) if isinstance(result,str) else result
        path = result[0]["file"]
        from urllib.parse import urlsplit
        parsed = urlsplit(path)
        assert not parsed.scheme and not parsed.netloc and parsed.path == "/v1/audio"
        urllib.request.urlretrieve(base+path, "/content/melodiva.wav")
        break
    time.sleep(2)
else:
    raise TimeoutError("Generation timed out")
from IPython.display import Audio, display
display(Audio(filename="/content/melodiva.wav"))


In [ ]:
from google.colab import files
files.download("/content/melodiva.wav")


In [ ]:
server.terminate()
server.wait(timeout=30)
log.close()
